# Notebook 09 - Greedy/VW variants and first RL checks

**Objetivo.** Probar variantes de greedy con super-nodos VW y primeras curvas de reinforcement learning.

**Pregunta guia.** La compresion VW y RL tabular cambian la decision o solo reformulan el greedy miope?

**Lectura esperada.** Las figuras 1-2 son VW; las figuras 3-4 son consultas iniciales de RL.

**Formato.** Cada bloque sigue el mismo patron: contexto breve, parametros (`n`, `B`, `G`, `p`, `u`), calculo reproducible y salida interpretada cerca del codigo.

## Setup

In [ ]:
import os, sys, time
import math
from itertools import combinations
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(''))))

from augmented.core import indices_from_mask, mask_from_indices, test_result
from augmented.bayesian import bayesian_update_single_test
from augmented.greedy import greedy_myopic_simulate
from augmented.solver import solve_optimal_dapts
from augmented.rl_examples import (
    value_iteration, tabular_q_learning, q_learning_policy_value,
)

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

# Parte 1 — La idea de super-nodos (VW)

## Instancia

$n=6$, pools de tamaño $G=3$, presupuesto $B=2$. Ya se hizo una consulta sobre $\{0,2,4\}$; eso fija $S$ (los super-nodos posibles) y $V$ (los aún no tocados).

In [ ]:
n, G, B = 6, 3, 2
p_prior = [0.10, 0.15, 0.20, 0.08, 0.12, 0.25]
u = [4.0, 6.0, 3.0, 5.0, 7.0, 4.0]

pool1 = mask_from_indices([0, 2, 4])
z_true = mask_from_indices([2])
r1 = test_result(pool1, z_true)
p_post = bayesian_update_single_test(p_prior, pool1, r1, n)

S_idx = indices_from_mask(pool1, n)
V_idx = [i for i in range(n) if i not in S_idx]
print('S =', S_idx, '  V =', V_idx)

## Figura 1 — ¿VW reproduce al greedy miope?

Para cada pool posible comparamos su valor miope contra el valor que le asigna la formulación VW.

In [ ]:
def myopic_score(pool):
    util = sum(u[i] for i in pool)
    return util * math.prod(1 - p_post[i] for i in pool)

def vw_score(pool, mode):
    T = [i for i in pool if i in S_idx]
    U = [i for i in pool if i in V_idx]
    util = sum(u[i] for i in pool)
    pclear_U = math.prod(1 - p_post[i] for i in U)
    pclear_T = math.prod(1 - p_post[i] for i in T) if T else 1.0
    prob_T = pclear_T if mode == 'all-clear' else ((1 - pclear_T) if T else 1.0)
    return util * pclear_U * prob_T

universe = S_idx + V_idx
full, vwA, vwB = [], [], []
for size in range(1, G + 1):
    for pool in combinations(universe, size):
        full.append(myopic_score(pool))
        vwA.append(vw_score(pool, 'all-clear'))
        vwB.append(vw_score(pool, 'or-event'))

fig, ax = plt.subplots(figsize=(4.6, 4.3))
lim = max(full) * 1.05
ax.plot([0, lim], [0, lim], color='0.7', lw=1, zorder=0)
ax.scatter(full, vwB, s=30, color='#d1495b', label='VW or-event', zorder=2)
ax.scatter(full, vwA, s=30, color='#2e7d32', label='VW all-clear', zorder=3)
ax.set_xlabel('valor miope (pool completo)')
ax.set_ylabel('valor de la formulación VW')
ax.set_title('Figura 1')
ax.legend(frameon=False)
plt.tight_layout()
fig.savefig('07_vw_fig1.png', bbox_inches='tight', dpi=130)
plt.show()

In [ ]:
q = [1 - pi for pi in p_prior]
Bs = [2, 3]
greedy_vals, dp_vals = [], []
for Bx in Bs:
    g = 0.0
    for z in range(1 << n):
        w = 1.0
        for i in range(n):
            w *= p_prior[i] if (z >> i) & 1 else q[i]
        _, _, util_z = greedy_myopic_simulate(p_prior, u, Bx, G, z)
        g += w * util_z
    dp, _ = solve_optimal_dapts(p_prior, u, Bx, G)
    greedy_vals.append(g)
    dp_vals.append(dp)

x = np.arange(len(Bs))
fig, ax = plt.subplots(figsize=(5.0, 3.6))
ax.bar(x - 0.18, greedy_vals, 0.36, label='greedy / VW', color='#e08a3c')
ax.bar(x + 0.18, dp_vals, 0.36, label='DP óptimo', color='#3a6ea5')
for i in range(len(Bs)):
    ax.annotate(f'+{dp_vals[i] - greedy_vals[i]:.2f}',
                (x[i], max(dp_vals[i], greedy_vals[i]) + 0.04),
                ha='center', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([f'B = {b}' for b in Bs])
ax.set_ylabel('utilidad esperada')
ax.set_title('Figura 2')
ax.legend(frameon=False)
plt.tight_layout()
fig.savefig('07_vw_fig2.png', bbox_inches='tight', dpi=130)
plt.show()

Primeras consultas de Reinforcement Learning



In [ ]:
n_rl, B_rl, G_rl = 4, 2, 3
p_rl = [0.15, 0.25, 0.10, 0.30]
u_rl = [3.0, 5.0, 4.0, 6.0]
dp_opt, _ = solve_optimal_dapts(p_rl, u_rl, B_rl, G_rl)

checkpoints = [50, 150, 400, 1000, 3000, 8000, 20000]
seeds = range(6)
curve = []
for ep in checkpoints:
    vals = [q_learning_policy_value(p_rl, u_rl, B_rl, G_rl,
            tabular_q_learning(p_rl, u_rl, B_rl, G_rl,
                               num_episodes=ep, epsilon=0.5, seed=s))
            for s in seeds]
    curve.append(sum(vals) / len(vals))

fig, ax = plt.subplots(figsize=(5.4, 3.8))
ax.axhline(dp_opt, color='#d1495b', ls='--', lw=1.2, label=f'óptimo DP = {dp_opt:.2f}')
ax.plot(checkpoints, curve, 'o-', color='#2e7d32', label='Q-learning (media 6 corridas)')
ax.set_xscale('log')
ax.set_xlabel('episodios de entrenamiento')
ax.set_ylabel('utilidad esperada de la política')
ax.set_title('Figura 3')
ax.legend(frameon=False, loc='lower right')
plt.tight_layout()
fig.savefig('07_vw_fig3.png', bbox_inches='tight', dpi=130)
plt.show()

In [ ]:
ns = list(range(2, 10))
times = []
for nn in ns:
    pp = [0.10 + 0.03 * i for i in range(nn)]
    uu = [1.0 + i for i in range(nn)]
    t0 = time.time()
    value_iteration(pp, uu, 2, 3)
    times.append(time.time() - t0)

fit_n = np.array([nn for nn in ns if nn >= 5])
fit_t = np.array([t for nn, t in zip(ns, times) if nn >= 5])
a, b = np.polyfit(fit_n, np.log(fit_t), 1)
ext_n = np.arange(5, 17)
ext_t = np.exp(a * ext_n + b)

fig, ax = plt.subplots(figsize=(5.4, 3.8))
ax.scatter(ns, [max(t, 1e-4) for t in times], s=34, color='#3a6ea5',
           zorder=3, label='solución exacta (medido)')
ax.plot(ext_n, ext_t, '--', color='#9aa0a6', label='extrapolación')
ax.axvline(14, color='#d1495b', lw=1.2)
ax.text(13.7, ext_t.max(), 'pared n=14', color='#d1495b',
        rotation=90, va='top', ha='right', fontsize=9)
ax.set_yscale('log')
ax.set_xlabel('n (individuos)')
ax.set_ylabel('tiempo de resolución exacta (s)')
ax.set_title('Figura 4')
ax.legend(frameon=False, loc='upper left')
plt.tight_layout()
fig.savefig('07_vw_fig4.png', bbox_inches='tight', dpi=130)
plt.show()